<a href="https://colab.research.google.com/github/VaibhavGoyal9805/Transformer-From-Scratch/blob/main/notebooks/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer From Scratch - Colab Training

This notebook runs the fully optimized Transformer model on Google Colab's free GPUs (T4). It mounts your Google Drive so that your trained model checkpoints are saved permanently.

### 1. Mount Google Drive
Run this cell to connect your Google Drive. This ensures your training progress isn't lost when the Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Setup Project
Clone the repository into your Drive (or pull the latest changes if it's already there) and install dependencies.

In [ ]:
import os

drive_path = '/content/drive/MyDrive/Transformer-From-Scratch'

if not os.path.exists(drive_path):
    %cd /content/drive/MyDrive
    !git clone https://github.com/VaibhavGoyal9805/Transformer-From-Scratch.git
else:
    print("Repository already exists in Drive. Pulling latest changes...")
    %cd {drive_path}
    !git pull origin main

%cd {drive_path}
!pip install -r requirements.txt

### 3. Start Optimized Training
This runs the 15-epoch training with the optimized architecture (15M parameters). On a Colab T4 GPU, this should be significantly faster than running locally.

In [ ]:
import sys
sys.path.insert(0, 'src')
from trainer import train, plot_training_curves
from generate import generate_text
import torch

print('='*60)
print('OPTIMIZED TRAINING ON COLAB GPU')
print('='*60)

# Ensure CUDA is available
if not torch.cuda.is_available():
    print("WARNING: CUDA is not available. Please go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
    device = 'cpu'
else:
    device = 'cuda'

model, tok, hist = train(
    epochs=15,
    d_model=256,
    n_heads=8,
    d_ff=1024,
    n_layers=6,
    dropout=0.1,
    seq_len=128,
    batch_size=64,  # Increased batch size since Colab T4 has 16GB VRAM
    warmup_steps=500,
    weight_decay=0.01,
    label_smoothing=0.1,
    save_every=5,
    device_str=device,
)

plot_training_curves(hist)

print('\n' + '='*60)
print('SAMPLE GENERATIONS:')
print('='*60)
for m in ['greedy','top_k','temperature']:
    t = generate_text(model, tok, prompt='To be or not', max_tokens=50, method=m, temperature=0.8, top_k=40)
    print(f'\n[{m.upper()}]: {t}')


OPTIMIZED TRAINING ON COLAB GPU
[Trainer] Using device: cuda
[Trainer] Loading data...
[Trainer] Vocab size: 13,335
[Trainer] Train batches: 3,696  Val batches: 409
[Trainer] Model parameters: 11,579,927

[Trainer] Starting training for 15 epochs...
------------------------------------------------------------
  Epoch   1/15 │ Train Loss 6.9099 │ Val Loss 6.9559 │ PPL 1049.37 │ LR 1.03e-03 │ 853.4s
  Epoch   2/15 │ Train Loss 6.8858 │ Val Loss 6.9584 │ PPL 1051.93 │ LR 7.27e-04 │ 849.9s
